### And welcome to Week 4, Day 3 - more LangGraph..

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
# from langchain_openai import ChatOpenAI
from typing import TypedDict


In [ ]:
# Our favorite first step! Crew was doing this for us, by the way.
load_dotenv(override=True)


### First, let's go set up LangSmith!

https://langsmith.com

### Next, here is a useful function in LangChain community:

In [ ]:
# from langchain_community.utilities import GoogleSerperAPIWrapper

# serper = GoogleSerperAPIWrapper()
# serper.run("What is the capital of France?")

from langchain_community.tools import BraveSearch
tool_search = BraveSearch()

### Now here is a LangChain wrapper class for converting functions into Tools

In [ ]:
# from langchain.agents import Tool

# tool_search =Tool(
#         name="search",
#         func=serper.run,
#         description="Useful for when you need more information from an online search"
#     )



### Now we can try out the tool the langchain way

In [ ]:
tool_search.invoke("What is the capital of France?")

### And now let's write a tool ourselves

We'll pick a familiar one

In [ ]:
! uv add apprise

In [ ]:
! pip install --upgrade apprise

In [ ]:
import apprise
from langchain_core.tools import BaseTool
from typing import Optional

class NtfyAppriseTool(BaseTool):
    name: str = "send_ntfy_notification"
    description: str = (
        "Sends a push notification to user. "
        "Use this when you need to alert a user about task completion, errors, or important updates. "
        "Input should be a string containing the message to send. "
        "Optional: You can include a priority tag like 'priority:high' at the start of the message."
    )
    
    # Configuration: Replace with your actual Ntfy URL
    # Format: ntfy://server/topic?priority=1&tags=rocket
    # ntfy_url: str = "ntfy.sh/my-unique-topic" 
    ntfy_url: str = os.getenv("NTFY_URL")
    # If self-hosted: "http://localhost:8080/my-topic"

    def _run(self, message: str) -> str:
        """
        Run the tool to send a notification.
        
        Args:
            message (str): The message content to send.
            
        Returns:
            str: Success or failure message.
        """
        try:
            # Initialize Apprise
            apobj = apprise.Apprise()
            
            # Add the Ntfy URL
            if not apobj.add(self.ntfy_url):
                return "Invalid Ntfy URL format."
            
            # Optional: Parse priority from message if the user includes a prefix
            # Example input: "priority:high Backup finished"
            # priority = 3  # Default: Normal
            # if message.startswith("priority:high"):
            #     priority = 4
            #     message = message.replace("priority:high", "").strip()
            # elif message.startswith("priority:low"):
            #     priority = 1
            #     message = message.replace("priority:low", "").strip()
                
            # Send the notification
            # Apprise handles the HTTP request to Ntfy
            result = apobj.notify(
                body=message,
                title="LangChain Agent Alert",
                # priority=priority
            )
            
            if result:
                return f"Notification sent successfully to {self.ntfy_url}"
            else:
                return "Failed to send notification. Check Ntfy server status."
                
        except Exception as e:
            return f"Error sending notification: {str(e)}"

    async def _arun(self, message: str) -> str:
        """Async implementation of the tool."""
        # Apprise is synchronous by default, but we can run it in a thread
        import asyncio
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self._run, message)


# Initialize the tool
tool_push = NtfyAppriseTool()
tool_push.invoke("Hello, me")


In [ ]:
# pushover_token = os.getenv("PUSHOVER_TOKEN")
# pushover_user = os.getenv("PUSHOVER_USER")
# pushover_url = "https://api.pushover.net/1/messages.json"

# def push(text: str):
#     """Send a push notification to the user"""
#     requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

In [ ]:
# tool_push = Tool(
#         name="send_push_notification",
#         func=push,
#         description="useful for when you want to send a push notification"
#     )

# tool_push.invoke("Hello, me")

### Back to the Graph from yesterday

One small change - using TypedDict instead of BaseModel for the State object

When we implement tools, we always need to make 2 changes to the code:

1. Changes to provide the tools to OpenAI in json when we make the call

2. Changes to handle the results back: look for the model staying that the finish_reason=="tool_calls" and then retrieve the call, run the function, provide the results.

### Bring them together

In [ ]:
tools = [tool_search, tool_push]

In [ ]:
# Step 1: Define the State object
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# Step 2: Start the Graph Builder with this State class
graph_builder = StateGraph(State)

In [ ]:
# This is different:

# llm = ChatOpenAI(model="gpt-4o-mini")
# llm_with_tools = llm.bind_tools(tools)

In [ ]:
# pip install -U langchain-anthropic
# export ANTHROPIC_API_KEY="your-api-key"

from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-haiku-4-5",#claude-haiku-4-5-20251001
    # temperature=,
    # max_tokens=,
    # timeout=,
    # max_retries=,
    # base_url="...",
    # Refer to API reference for full list of parameters
)

In [ ]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
llm.invoke("hi!")

In [ ]:
# Step 3: Create a Node


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

In [ ]:
# Step 4: Create Edges


graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")

# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
# Step 5: Compile the Graph
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### That's it! And, let's do this:

In [ ]:
graph

In [ ]:
def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

## OK it's time to add Memory!

### BUT WAIT!

We have this whole Graph maintaining the state and appending to the state.

Why isn't this handling memory?

### This is a crucial point for understanding LangGraph

> A super-step can be considered a single iteration over the graph nodes. Nodes that run in parallel are part of the same super-step, while nodes that run sequentially belong to separate super-steps.


One "Super-Step" of the graph represents one invocation of passing messages between agents.

In idomatic LangGraph, you call invoke to run your graph for each super-step; for each interaction.

The reducer handles state updates automatically within one super-step, but not between them.

That is what checkpointing achieves.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
# Steps 1 and 2
graph_builder = StateGraph(State)


# Step 3
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    print(state)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Step 4
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# Step 5
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()

In [ ]:
graph.get_state(config)

In [ ]:
# Most recent first

list(graph.get_state_history(config))

### LangGraph gives you tools to set the state back to a prior point in time, to branch off:

```
config = {"configurable": {"thread_id": "1", "checkpoint_id": ...}}
graph.invoke(None, config=config)
```

And this allows you to build stable systems that can be recovered and rerun from any prior checkpoint.

### And now let's store in SQL

### And this is the power of LangGraph.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

In [ ]:
# Steps 1 and 2
graph_builder = StateGraph(State)


# Step 3
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    print(state)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Step 4
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

# Step 5
graph = graph_builder.compile(checkpointer=sql_memory)
display(Image(graph.get_graph().draw_mermaid_png()))
 

In [ ]:
config = {"configurable": {"thread_id": "3"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()